# Iris Flower Classification
Run this notebook from the extracted project folder. Install requirements first.


In [ ]:
"""Train and evaluate an Iris classifier. Run: python train_iris.py"""
from pathlib import Path
import json
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

BASE = Path.cwd()
FEATURES = ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']

def main():
    df = pd.read_csv(BASE / 'Iris.csv')
    if df[FEATURES + ['Species']].isna().any().any():
        raise ValueError('Missing measurements or species labels.')
    X, y = df[FEATURES], df['Species']
    # Id identifies a row; it must not be used as a predictive measurement.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
    # Scaling is learned inside each fold, preventing validation data leakage.
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    report = classification_report(y_test, predictions, output_dict=True, zero_division=0)
    metrics = {'model': 'StandardScaler + KNN (k=5)', 'seed': 42,
               'training_samples': len(X_train), 'test_samples': len(X_test),
               'test_accuracy': accuracy_score(y_test, predictions),
               'cv_accuracy_scores': scores.tolist(), 'cv_mean': scores.mean(),
               'cv_std': scores.std(), 'classification_report': report}
    (BASE / 'metrics.json').write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))
    results = X_test.copy()
    results.insert(0, 'Id', df.loc[X_test.index, 'Id'])
    results['Actual'] = y_test
    results['Predicted'] = predictions
    results.to_csv(BASE / 'test_predictions.csv', index=False)
    ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap='Blues',
        display_labels=['setosa', 'versicolor', 'virginica'], colorbar=False)
    plt.title('Iris classification: held-out test set')
    plt.tight_layout()
    plt.savefig(BASE / 'confusion_matrix.png', dpi=170)
    plt.close()
    for species, group in df.groupby('Species'):
        plt.scatter(group.PetalLengthCm, group.PetalWidthCm, label=species, alpha=.75)
    plt.xlabel('Petal length (cm)'); plt.ylabel('Petal width (cm)')
    plt.title('Iris species by petal measurements'); plt.legend(); plt.tight_layout()
    plt.savefig(BASE / 'petal_measurements.png', dpi=170); plt.close()
    joblib.dump(model, BASE / 'iris_model.joblib')
    sample = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=FEATURES)
    print('Example prediction:', model.predict(sample)[0])

if __name__ == '__main__':
    main()


# Iris Flower Classification

## Introduction
The objective was to train a machine learning model to classify an Iris flower as setosa, versicolor, or virginica from its sepal and petal measurements. This is supervised multiclass classification: the model learns from examples whose species labels are already known.

## Dataset and preparation
The supplied Iris.csv contains 150 flowers, with 50 examples of each species. The four input features are sepal length, sepal width, petal length, and petal width, measured in centimetres. Species is the target label. No missing values were found. The Id column was excluded because it identifies records rather than describing flower measurements.

A stratified 80:20 split produced 120 training records and 30 test records, preserving class balance (40 training and 10 test records per species). The random seed was 42 so the split can be reproduced.

## Model and method
Scikit-learn was used to build a pipeline containing StandardScaler and a K-Nearest Neighbours classifier with k=5. KNN predicts the majority species among the five nearest training flowers. Standardisation gives each measurement a comparable scale so that differences in numeric spread do not dominate distance calculations.

Five-fold stratified cross-validation was performed only on the training data. In each fold, four parts were used to fit the pipeline and one part to validate it. Scaling was fitted separately within each fold to prevent leakage. The fixed model was then fitted on all 120 training records and evaluated on the untouched 30-record test set. No test-driven parameter tuning was performed.

## Results
The model correctly classified 28 out of 30 test flowers.

| Measure | Result |
|---|---:|
| Test accuracy | 93.33% |
| Training cross-validation mean accuracy | 95.83% |
| Cross-validation standard deviation | 2.64 percentage points |
| Macro F1-score | 0.9327 |

| Species | Precision | Recall | F1-score | Test samples |
|---|---:|---:|---:|---:|
| Setosa | 1.0000 | 1.0000 | 1.0000 | 10 |
| Versicolor | 0.8333 | 1.0000 | 0.9091 | 10 |
| Virginica | 1.0000 | 0.8000 | 0.8889 | 10 |

Confusion matrix (rows = actual species; columns = predicted species):

| Actual / Predicted | Setosa | Versicolor | Virginica |
|---|---:|---:|---:|
| Setosa | 10 | 0 | 0 |
| Versicolor | 0 | 10 | 0 |
| Virginica | 0 | 2 | 8 |

![Confusion matrix](confusion_matrix.png)

The two errors were virginica flowers predicted as versicolor. The petal scatter plot shows overlap between these two species, which helps explain the errors; setosa is more clearly separated.

![Petal measurements](petal_measurements.png)

## Basic classification concepts
- Features: measured inputs used to make predictions.
- Target: the species the model must predict.
- Training data: labelled examples used to fit the model.
- Test data: held-out examples used to assess generalisation.
- Accuracy: correct predictions divided by all predictions (28/30).
- Precision: among predictions for a species, the fraction that are correct.
- Recall: among actual flowers of a species, the fraction found correctly.
- F1-score: the harmonic mean of precision and recall, balancing both measures.
- Overfitting: learning training-specific details that do not generalise well to unseen examples. Held-out testing and cross-validation help assess this risk.

## Conclusion and limitations
The task was completed using the supplied data and Scikit-learn. A scaled KNN classifier achieved 93.33% accuracy on the held-out test set and 95.83% average training cross-validation accuracy. These results demonstrate effective classification on this small dataset, but 30 test samples do not establish performance on all real-world flowers. The system recognises only the three trained species and assumes correctly measured inputs in centimetres. Future work could compare classifiers and assess them through a separate validation strategy while preserving a final untouched test set.
